<a href="https://colab.research.google.com/github/galfaro23/STEM_LADAINA/blob/Grettel/4_Codigo_Limpieza/Codigo_EstandarizacionDeTitulos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Este codigo se aplica solo a titulos en español

In [22]:
import pandas as pd
ruta_csv = '/content/Dataset_DivididoPorIdioma.csv'
# Read the CSV, specifying the separator and that the first row is the header.
# Keep on_bad_lines='skip' to handle problematic rows.
df = pd.read_csv(ruta_csv, sep=',', on_bad_lines='skip', header=0)
print(df.head())

         Fuente            Categoría  \
0  Computrabajo  Software Developers   
1  Computrabajo  Software Developers   
2  Computrabajo  Software Developers   
3  Computrabajo       Web Developers   
4  Computrabajo       Web Developers   

                                                       Título  \
0                                      Analista desarrollador   
1                                   Programador Sistemas .Net   
2  Técnico en Soporte Informático San José Cartago y Alajuela   
3                            Desarrollador Senior TI (Oracle)   
4                              Técnico en Soporte informático   

                                                            Empresa  \
0  FACTOR K QUINIENTOS SEIS SC SOCIEDAD DE RESPONSABILIDAD LIMITADA   
1                                    Architect Technology Solutions   
2                                                 SOPORTEXPERTO.COM   
3                                                           Asembis   
4            

In [23]:
print(df)

             Fuente             Categoría  \
0      Computrabajo   Software Developers   
1      Computrabajo   Software Developers   
2      Computrabajo   Software Developers   
3      Computrabajo        Web Developers   
4      Computrabajo        Web Developers   
...             ...                   ...   
10882      RemoteOK  Ingenieros de ventas   
10883      RemoteOK  Ingenieros de ventas   
10884      RemoteOK  Ingenieros de ventas   
10885      RemoteOK  Ingenieros de ventas   
10886      RemoteOK  Ingenieros de ventas   

                                                            Título  \
0                                           Analista desarrollador   
1                                        Programador Sistemas .Net   
2       Técnico en Soporte Informático San José Cartago y Alajuela   
3                                 Desarrollador Senior TI (Oracle)   
4                                   Técnico en Soporte informático   
...                                    

In [41]:
!python -m spacy download es_core_news_md

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 MB 15.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [44]:
# -*- coding: utf-8 -*-
"""
Pipeline extendido:
- Persistencia de diccionarios (JSON/YAML)
- Detección de roles emergentes/typos con ML (scikit-learn) o fallback a difflib
- Clasificación automática df["categoria"]
"""

import os
import json
import re
import unicodedata
from typing import Dict, List, Tuple, Optional
from collections import Counter
from difflib import get_close_matches

import pandas as pd

# ============
# Normalizador
# ============
def normalize_text(s: str) -> str:
    if s is None or (isinstance(s, float) and pd.isna(s)):  # por si vienen NaN
        return ""
    s = str(s).lower().strip()
    s = "".join(c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn")
    s = re.sub(r"[^\w\s+&]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


# ======================================================
# 1) DICCIONARIOS BASE (puedes reemplazarlos por los tuyos)
#    — aquí pongo una versión resumida y limpia —
# ======================================================
seniority_dict = {
    "jr": "junior", "junior": "junior", "trainee": "junior", "entry": "junior",
    "ssr": "semisenior", "semi": "semisenior", "mid": "semisenior",
    "sr": "senior", "senior": "senior", "avanzado": "senior",
    "especialista": "senior", "experimentado": "senior"
}

leadership_dict = {
    "lead": "lead", "lider": "lead", "líder": "lead",
    "manager": "manager", "mánager": "manager", "gerente": "manager",
    "head": "head", "chief": "chief",
    "director": "director", "coordinador": "coordinator",
    "supervisor": "supervisor", "responsable": "supervisor"
}

modalidad_dict = {
    "remoto": "remoto", "remote": "remoto", "teletrabajo": "remoto", "home office": "remoto",
    "hibrido": "hibrido", "hybrid": "hibrido",
    "presencial": "presencial", "onsite": "presencial", "en sitio": "presencial"
}

jornada_dict = {
    "full time": "tiempo completo", "jornada completa": "tiempo completo",
    "part time": "medio tiempo", "medio tiempo": "medio tiempo",
    "nocturno": "nocturno", "diurno": "diurno", "rotativo": "rotativo"
}

contrato_dict = {
    "temporal": "temporal", "freelance": "freelance",
    "intern": "intern", "practicante": "intern", "pasante": "intern"
}

# Grupos semánticos de puestos (puedes ampliarlos con lo que ya tienes)
grupos_puestos = {
    "developer": ["desarrollador", "programador", "dev", "coder", "developer", "software developer"],
    "data scientist": ["cientifico de datos", "data scientist", "cientifica de datos", "científica de datos"],
    "data analyst": ["analista de datos", "data analyst"],
    "qa": ["qa", "tester", "quality assurance", "testing", "pruebas"],
    "devops": ["devops", "infraestructura", "sre"],
    "ingeniero": ["ingeniero", "engineer", "ing", "engineering"],
    "tecnico": ["tecnico", "technician", "técnico", "tecnica", "tecnologo", "tecnología"],  # ojo con 'tecnología' si no quieres mapearlo
    "diseñador": ["diseñador", "designer", "design", "ux", "ui", "uxui", "diseño"],
    "soporte": ["soporte", "helpdesk", "help desk", "service desk"],
    "administrativo": ["asistente", "auxiliar", "administrativo", "secretario", "recepcionista"],
    "gerente": ["gerente", "manager", "director", "head"],
    "vendedor": ["vendedor", "sales", "comercial", "ejecutivo", "account manager"],
    "seguridad": ["seguridad", "oficial seguridad", "guardia", "vigilante"],
    "enfermero": ["enfermero", "nurse", "enfermera"],
    "contador": ["contador", "accountant"],
}

# Diccionario plano {variante: canon}
puesto_dict = {var: canon for canon, variantes in grupos_puestos.items() for var in variantes}


# ======================================================
# 2) PERSISTENCIA: GUARDAR/CARGAR JSON y YAML (si existe)
# ======================================================
def save_dicts(base_dir: str, dicts: Dict[str, Dict[str, str]]) -> None:
    """
    Guarda cada diccionario en base_dir/{nombre}.json y, si PyYAML está disponible,
    también en base_dir/{nombre}.yaml
    """
    os.makedirs(base_dir, exist_ok=True)

    # Guardar JSON
    for name, d in dicts.items():
        path_json = os.path.join(base_dir, f"{name}.json")
        with open(path_json, "w", encoding="utf-8") as f:
            json.dump(d, f, ensure_ascii=False, indent=2)

    # Guardar YAML si es posible
    try:
        import yaml  # type: ignore
        for name, d in dicts.items():
            path_yaml = os.path.join(base_dir, f"{name}.yaml")
            with open(path_yaml, "w", encoding="utf-8") as f:
                yaml.safe_dump(d, f, allow_unicode=True, sort_keys=True)
    except Exception:
        # Si no hay pyyaml, no hacemos nada extra.
        pass


def load_dict(path: str) -> Dict[str, str]:
    """
    Carga un diccionario desde JSON o YAML según extensión.
    """
    ext = os.path.splitext(path)[1].lower()
    if ext == ".json":
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    elif ext in (".yml", ".yaml"):
        import yaml  # necesitarás pyyaml
        with open(path, "r", encoding="utf-8") as f:
            return yaml.safe_load(f) or {}
    else:
        raise ValueError(f"Extensión no soportada: {ext}")


# ======================================================
# 3) LIMPIEZA Y CANONICALIZACIÓN DEL TÍTULO
# ======================================================
ruido_general = {"de","en","para","con","y","o","el","la","los","las","un","una","por","del","al"}
palabras_ruido = set().union(
    ruido_general,
    seniority_dict.keys(),
    leadership_dict.keys(),
    modalidad_dict.keys(),
    jornada_dict.keys(),
    contrato_dict.keys()
)

def limpiar_titulo_puro(titulo: str) -> str:
    s = normalize_text(titulo)
    if not s:
        return ""
    tokens = [t for t in s.split() if t not in palabras_ruido]
    # mapear sinónimos a canónicos
    mapped = [puesto_dict.get(t, t) for t in tokens]
    # quitar duplicados consecutivos
    s2 = re.sub(r"\b(\w+)( \1\b)+", r"\1", " ".join(mapped))
    return s2.strip()


# ======================================================
# 4) MODELO ML PARA ROLES EMERGENTES / TYPOS
#    - Usa char n-grams TF-IDF + LogisticRegression si sklearn está disponible
#    - Si no, fallback a difflib para aproximación léxica
# ======================================================
class RoleDetector:
    def __init__(self, puesto_dict: Dict[str, str]):
        """
        Entrena con pares (variante -> canon). Si sklearn no está disponible, usa difflib.
        """
        self.puesto_dict = {normalize_text(k): v for k, v in puesto_dict.items()}
        self.canon_labels = sorted(set(self.puesto_dict.values()))
        self._ml_ready = False

        try:
            from sklearn.feature_extraction.text import TfidfVectorizer
            from sklearn.linear_model import LogisticRegression
            from sklearn.pipeline import Pipeline

            # dataset de entrenamiento: variantes como X, canon como y
            X = list(self.puesto_dict.keys())
            y = [self.puesto_dict[k] for k in X]

            # char n-grams: robusto a typos
            self.pipeline = Pipeline([
                ("tfidf", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5))),
                ("clf", LogisticRegression(max_iter=400, n_jobs=None))
            ])
            self.pipeline.fit(X, y)
            self._ml_ready = True

        except Exception:
            # no sklearn: usaremos difflib sobre el espacio de variantes
            self._variants = list(self.puesto_dict.keys())
            self.pipeline = None

    def predict(self, text: str) -> Tuple[str, float, str]:
        """
        Devuelve (canon_predicho, score_aprox, metodo)
        - score_aprox: probabilidad (si ML) o ratio de similitud (si difflib)
        """
        q = normalize_text(text)

        if not q:
            return ("", 0.0, "empty")

        # ML si está disponible
        if self._ml_ready and self.pipeline is not None:
            try:
                proba = self.pipeline.predict_proba([q])[0]
                classes = self.pipeline.classes_
                idx = proba.argmax()
                return (classes[idx], float(proba[idx]), "ml")
            except Exception:
                pass

        # Fallback: difflib sobre variantes -> luego mapear a canon
        matches = get_close_matches(q, self._variants, n=1, cutoff=0.6)
        if matches:
            variant = matches[0]
            canon = self.puesto_dict.get(variant, variant)
            # ratio aproximado (no tenemos el ratio directo aquí; estimación burda)
            ratio = 0.75
            return (canon, ratio, "difflib")

        return ("", 0.0, "none")


# ======================================================
# 5) CLASIFICACIÓN A CATEGORÍAS (taxonomía alta)
#    Mapea el "canon" a una categoría de negocio.
# ======================================================
canon_to_categoria = {
    # Tecnología
    "developer": "Tecnología",
    "data scientist": "Tecnología",
    "data analyst": "Tecnología",
    "qa": "Tecnología",
    "devops": "Tecnología",
    "ingeniero": "Tecnología",
    "tecnico": "Tecnología",
    "diseñador": "Tecnología",
    "soporte": "Tecnología",
    "gerente": "Tecnología",   # si manejas gerencias fuera de tech, ajusta este mapping
    # Comercial
    "vendedor": "Comercial/Ventas",
    # Seguridad
    "seguridad": "Seguridad",
    # Salud
    "enfermero": "Salud",
    # Finanzas
    "contador": "Finanzas",
    # Default
}

def clasificar_puesto(canon: str) -> str:
    return canon_to_categoria.get(canon, "Otros")


# ======================================================
# 6) INTEGRACIÓN CON DATAFRAME
#    - df["titulo_es_clean"]: limpieza + mapeo básicos
#    - df["rol_canonico_ml"]: ML/fallback para detectar rol aunque no esté en diccionario
#    - df["categoria"]: a partir del canon detectado
# ======================================================
def aplicar_pipeline(df: pd.DataFrame,
                     col_titulo: str = "titulo_es",
                     persist_dir: Optional[str] = "diccionarios") -> pd.DataFrame:
    """
    Aplica:
      - limpieza -> titulo_es_clean
      - persistencia de diccionarios (opcional) en 'persist_dir'
      - detector de rol (ML o difflib) -> rol_canonico_ml
      - clasificación -> categoria
    """
    assert col_titulo in df.columns, f"No existe la columna '{col_titulo}'"

    # 1) Guardar diccionarios (opcional pero recomendado)
    if persist_dir:
        save_dicts(persist_dir, {
            "seniority_dict": seniority_dict,
            "leadership_dict": leadership_dict,
            "modalidad_dict": modalidad_dict,
            "jornada_dict": jornada_dict,
            "contrato_dict": contrato_dict,
            "puesto_dict": puesto_dict
        })

    # 2) Limpiar títulos
    df = df.copy()
    df["titulo_es_clean"] = df[col_titulo].map(limpiar_titulo_puro)

    # 3) Entrenar detector (rápido, sobre diccionario)
    detector = RoleDetector(puesto_dict)

    # 4) Predecir canon para cada fila (aun con typos/roles nuevos)
    preds, scores, methods = [], [], []
    for txt in df["titulo_es_clean"].fillna(""):
        canon, score, method = detector.predict(txt)
        preds.append(canon)
        scores.append(score)
        methods.append(method)

    df["rol_canonico_ml"] = preds
    df["rol_confianza"] = scores
    df["rol_metodo"] = methods

    # 5) Categoría a partir del canon
    df["categoria"] = df["rol_canonico_ml"].map(clasificar_puesto)

    return df


# ======================================================
# 7) EJEMPLO DE USO (descomenta para correr)
# ======================================================
# Supongamos que ya tienes df con columna 'titulo_es'
# df = pd.read_csv("tu_archivo.csv")
df = aplicar_pipeline(df, col_titulo="titulo_es", persist_dir="diccionarios")
# Guarda
df.to_csv("datos_limpios_con_categoria.csv", index=False, encoding="utf-8")
print("✅ Listo: 'datos_limpios_con_categoria.csv'")
# Muestra rápida
display(df[["titulo_es", "titulo_es_clean", "rol_canonico_ml", "rol_confianza", "rol_metodo", "categoria"]].head(20))


✅ Listo: 'datos_limpios_con_categoria.csv'


,titulo_es,titulo_es_clean,rol_canonico_ml,rol_confianza,rol_metodo,categoria
0,Analista desarrollador,analista developer,developer,0.210848,ml,Tecnología
1,NaN,,,0.000000,empty,Otros
2,Técnico en Soporte Informático San José Cartago y Alajuela,tecnico soporte informatico san jose cartago alajuela,tecnico,0.170633,ml,Tecnología
3,Desarrollador Senior TI (Oracle),developer ti oracle,developer,0.303157,ml,Tecnología
4,Técnico en Soporte informático,tecnico soporte informatico,tecnico,0.169974,ml,Tecnología
5,Desarrollador de puntos de venta,developer puntos venta,developer,0.246222,ml,Tecnología
6,Técnico de Soporte Informático,tecnico soporte informatico,tecnico,0.169974,ml,Tecnología
7,Desarrollador comercial,developer vendedor,developer,0.215626,ml,Tecnología
8,Desarrollador Full Stack,developer full stack,developer,0.295651,ml,Tecnología
9,"Desarrollador WEB, Ecommerce y Plataformas B2B",developer web ecommerce plataformas b2b,developer,0.256774,ml,Tecnología
